# CodeAlpha Data Science Internship
## Task 1: Iris Flower Classification

### Project Objective:
Train a machine learning model to classify iris flower species (*Setosa, Versicolor, and Virginica*) based on sepal and petal measurements using Scikit-Learn.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

%matplotlib inline
sns.set_theme(style="whitegrid")

### 1. Load Dataset & Inspection

In [ ]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df.head()

In [ ]:
print('Dataset Shape:', df.shape)
df.describe().T

In [ ]:
df['species'].value_counts()

### 2. Exploratory Data Analysis & Visualizations

In [ ]:
sns.pairplot(df, hue='species', markers=['o', 's', 'D'], palette='viridis')
plt.suptitle('Iris Feature Pairplot by Species', y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.drop(columns=['species']).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

### 3. Data Splitting & Feature Scaling

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Training samples: {len(X_train)}, Testing samples: {len(X_test)}')

### 4. Model Training & Benchmarking

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Support Vector Machine': SVC(kernel='linear', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

benchmarks = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    cv = cross_val_score(model, X_train_scaled, y_train, cv=5)
    benchmarks.append({
        'Model': name,
        'Test Accuracy (%)': round(acc * 100, 2),
        'CV Mean (%)': round(cv.mean() * 100, 2),
        'CV Std (%)': round(cv.std() * 100, 2)
    })

results_df = pd.DataFrame(benchmarks)
results_df

### 5. Detailed Best Model Evaluation (Support Vector Machine)

In [ ]:
best_model = models['Support Vector Machine']
y_pred_best = best_model.predict(X_test_scaled)

print('Classification Report:\n')
print(classification_report(y_test, y_pred_best, target_names=iris.target_names))

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title('Confusion Matrix - Support Vector Machine')
plt.xlabel('Predicted Species')
plt.ylabel('Actual Species')
plt.show()

### 6. Real-time Inference on Unseen Samples

In [ ]:
samples = np.array([
    [5.1, 3.5, 1.4, 0.2],  # Expected: setosa
    [6.0, 2.7, 4.0, 1.3],  # Expected: versicolor
    [6.5, 3.0, 5.5, 2.0]   # Expected: virginica
])

sample_preds = best_model.predict(scaler.transform(samples))
for i, (sample, pred) in enumerate(zip(samples, sample_preds)):
    print(f'Sample {i+1} {sample.tolist()} -> Predicted Class: {iris.target_names[pred]}')